# Shift-encoder Fourier ablation

Local comparison of the four runs in `shift_encoder_ablation`. Tables summarize the final five validation measurements; curves use the full MLflow history. Run from the repository root.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path('results/ablation/shift_encoder_ablation')
if not root.exists():
    root = Path('shift_encoder_ablation')
runs = pd.read_csv(root / 'runs.csv')
history = pd.read_csv(root / 'metric_history.csv')
labels = {
    'shift-ablation-log-128': 'Log 128',
    'shift-ablation-log-256': 'Log 256',
    'shift-ablation-log-512': 'Log 512',
    'shift-ablation-linear-256': 'Linear 256',
}


## Final-five summary

Repeated checkpoints from one training run are useful for smoothing late-training noise, but they are not independent seeds or confidence intervals.

In [ ]:
metrics = ['val/shift', 'val/shift/h_mae_ppm', 'val/shift/c_mae_ppm',
           'val/fp_macro_mae', 'probe/maccs_macro_auroc']
rows = []
for run_name, group in history[history.metric.isin(metrics)].groupby('run_name'):
    row = {'run_name': labels.get(run_name, run_name)}
    for metric in metrics:
        values = (group[group.metric == metric].sort_values(['step', 'timestamp_ms'])
                  .drop_duplicates('step', keep='last').tail(5).value)
        row[metric] = values.mean()
    rows.append(row)
pd.DataFrame(rows).sort_values('run_name')


## Full learning curves

In [ ]:
plot_metrics = {
    'val/shift': 'Validation shift loss',
    'val/shift/h_mae_ppm': '¹H shift MAE (ppm)',
    'val/shift/c_mae_ppm': '¹³C shift MAE (ppm)',
    'probe/maccs_macro_auroc': 'MACCS macro AUROC',
}
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
for ax, (metric, title) in zip(axes.flat, plot_metrics.items()):
    for run_name, group in history[history.metric == metric].groupby('run_name'):
        group = group.sort_values('step')
        ax.plot((group.step + 1) / 1000, group.value, marker='o', label=labels.get(run_name, run_name))
    ax.set(title=title, xlabel='Optimizer steps (thousands)')
    ax.grid(alpha=0.25)
axes[0, 0].legend()
plt.show()


## Runtime and memory

In [ ]:
last = pd.read_csv(root / 'last_metrics.csv')
columns = ['run_name', 'status', 'duration_minutes', 'param_fourier_strategy',
           'param_num_fourier_freqs', 'gpu_peak_allocated_mb', 'gpu_peak_reserved_mb']
last[columns].sort_values('run_name')


## Reading the comparison

The three log-spaced variants are practically tied. Linear spacing is worse at every validation checkpoint for shift loss and H/C MAE, and its MACCS probe remains far below every log-spaced run. With only one seed, small differences among log variants are descriptive; the large, trajectory-wide linear-versus-log separation is the robust result.